# 🧠 Chain of Thought Prompting

**Welcome!** This notebook is your hands-on introduction to one of the most powerful ideas in prompt engineering: **Chain of Thought (CoT)** reasoning.

You'll discover that a single sentence added to a prompt can turn a confused, error-prone model into a careful, step-by-step problem solver. Pretty wild, right?

---

**How to use this notebook:**

- Cells marked **[RUN]** — just execute them, no changes needed.
- Cells marked **[TODO]** — you need to fill in some code before running.

> ⚠️ **Important:** Please select the **TORCH** kernel before starting.


## 1. Setup the environment and define utility functions

**[RUN]** The cells below install dependencies and load the model. No edits needed — just run them!


In [1]:
# @title Install Dependencies {display-mode: "form"}
# @markdown Run this cell first to install the required packages.
!pip install transformers accelerate datasets

In [2]:
# @title Load the Qwen Model {display-mode: "form"}
# @markdown Loads a small Qwen model for fast GPU inference.
import torch
import numpy as np
import random
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    dtype=torch.float32,
)
model.eval()

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [3]:
# @title Define the response generation logic {display-mode: "form"}

import uuid
import time
from IPython.display import display, HTML, Javascript
import html as html_lib


def generate_response(prompt, query_for_thinking=True):
    """
    Runs the model on a given prompt and returns the response text and elapsed time.
    """
    start = time.time()
    if query_for_thinking:
        prompt_inst = prompt.split("Question:")[0]
        prompt_q = "Output ONLY yes if this user prompt CONTAINS A REQUEST/AN INSTRUCTION to USE REASONING or THINK, otherwise output ONLY no. User prompt: " + prompt_inst
        resp, _ = generate_response(prompt_q, query_for_thinking=False)
        # print(f"cls response: {resp}")
        thinking_mode = "yes" in resp[-15:].lower()
    else:
        thinking_mode = True
    # print(f"{prompt=}, {thinking_mode=}, {query_for_thinking=}")
    inputs = tokenizer.apply_chat_template(
        [
            {
                "role": "system",
                "content": """Be concise in your answers. Clearly highlight what is your answer.""" ,
            },
            {"role": "user", "content": prompt},
        ],
        add_generation_prompt=True,
        enable_thinking=thinking_mode,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
        )
    response_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1] :],
        skip_special_tokens=True,
    )
    elapsed = time.time() - start
    return str(response_text), elapsed


In [4]:
# @title Define Display Utilities {display-mode: "form"}
# @markdown Styled Markdown helpers for rendering prompts and model responses
from IPython.display import display, Markdown, clear_output

def display_sample(question, answer):
    display(Markdown(f"**Question:** {question}\n\n**Answer:** {answer}"))

def display_qwen(prompt):
    """
    Displays the prompt and model response using Markdown to properly render LaTeX.
    """
    print("⏳ Generating response...")

    # --- Call the model ---
    response_text, elapsed = generate_response(prompt)

    clear_output(wait=True)

    # Display using Markdown for LaTeX rendering
    md_content = f"""### Prompt:
{prompt}

### Model Response:
{response_text}

*⏱ Generated in {elapsed:.2f} seconds*"""
    display(Markdown(md_content))


## 2. Chain of Thought (CoT) Prompting

### What's the big idea?

Imagine asking someone a hard math question. If you just say _"Answer this"_, they might guess. But if you say _"Think through it step by step"_, suddenly they slow down, reason carefully, and are much more likely to get it right.

**Chain of Thought prompting does exactly this for LLMs.** By nudging the model to reason explicitly before answering, we unlock dramatically better performance on complex tasks — especially math, logic, and multi-step problems.

We'll test this on **MATH500**, a popular benchmark of competition-level math word problems. Spoiler: the difference will be very noticeable! 🚀


**[RUN]** Let's start by loading the MATH500 dataset and peeking at a sample question and its answer.


In [5]:
train_split_main =  load_dataset("HuggingFaceH4/MATH-500", split="test")
print(f"Train split size: {len(train_split_main)}")

Train split size: 500


In [6]:
IDX = 29
display_sample(train_split_main[IDX]["problem"], train_split_main[IDX]["solution"])

**Question:** The Greek army contained two types of soldiers: the upper class and the lower class soldiers. If there were a total of 5 upper class soldiers, and 10 lower class soldiers in a certain part of Athens, and the battle of Thermopylae demands a force of 4 upper class soldiers and 8 lower class soldiers, how many different battalions can be sent?

**Answer:** There are $\binom{5}{4}$ different ways to choose 4 from 5 upper class soldiers. For each of these, there are $\binom{10}{8}$ ways to choose 8 lower class soldiers. The number of different battalions, then, is $\binom{5}{4}\cdot \binom{10}{8} = \boxed{225}$.

**[RUN]** Now let's ask our model to solve this question — no guidance, no hints, just the raw question (**zero-shot prompting**).


In [7]:
question = train_split_main[IDX]["problem"]
prompt = f"""Question: {question}"""

display_qwen(prompt)

### Prompt:
Question: The Greek army contained two types of soldiers: the upper class and the lower class soldiers. If there were a total of 5 upper class soldiers, and 10 lower class soldiers in a certain part of Athens, and the battle of Thermopylae demands a force of 4 upper class soldiers and 8 lower class soldiers, how many different battalions can be sent?

### Model Response:
The total number of different battalions that can be sent is 5 upper class soldiers + 10 lower class soldiers = 15 battalions.

*⏱ Generated in 9.03 seconds*

Hm, not great 😬. The model jumped straight to an answer without really thinking things through. This is what _pattern matching without reasoning_ looks like.

### ✏️ [TODO] Your turn — add a CoT trigger!

The fix is surprisingly simple. Complete the `cot_prompt` variable below with a short phrase that encourages the model to **think step by step** before answering.

> 💡 **Hint:** Think about how you'd ask a student to slow down and show their work. Even 5–6 words can be enough!


In [8]:
# 🎯 SOLUTION
cot_prompt = "Let's solve this problem step-by-step, think about how to approach it."
# 🎯 SOLUTION
prompt = f"""{cot_prompt}, Question:
{question}"""

display_qwen(prompt)


### Prompt:
Let's solve this problem step-by-step, think about how to approach it., Question:
The Greek army contained two types of soldiers: the upper class and the lower class soldiers. If there were a total of 5 upper class soldiers, and 10 lower class soldiers in a certain part of Athens, and the battle of Thermopylae demands a force of 4 upper class soldiers and 8 lower class soldiers, how many different battalions can be sent?

### Model Response:
<think>
Okay, let's see. The problem is about figuring out how many different battalions can be sent for the battle of Thermopylae. There are two types of soldiers: upper class and lower class. The total number of upper class soldiers is 5, and lower class is 10. But the battle requires 4 upper and 8 lower soldiers. So, the question is asking how many different battalions can be sent.

First, I need to understand what exactly constitutes a battalion. Since the problem mentions different battalions, I assume that each battalion is a group of soldiers. But the key here is that the battle requires a specific number of each type. So, the upper class soldiers available are 5, and lower class is 10. However, the battle needs 4 upper and 8 lower. Therefore, the number of battalions would be determined by how many of each type can be sent.

Wait, but maybe there's a trick here. Sometimes in these problems, you have to consider that each battalion can consist of multiple soldiers of the same type. But the problem says "different battalions," so maybe each battalion is a group of soldiers, and the question is how many different groups can be formed. But the problem states that the battle requires 4 upper and 8 lower soldiers. So, perhaps the answer is the number of ways to choose 4 upper and 8 lower soldiers, but since the problem says "how many different battalions can be sent," maybe it's just the number of possible combinations.

But let me think again. If each battalion is a group of soldiers, and the battle requires 4 upper and 8 lower, then the number of battalions would be the number of ways to choose 4 upper and 8 lower soldiers. However, since the upper class soldiers are only 5, and lower is 10, we can only form battalions with 4 upper and 8 lower. But wait, maybe the problem is simpler. Maybe the answer is just the number of possible battalions, which would be the number of ways to select 4 upper and 8 lower soldiers. But since the upper class soldiers are only 5, we can't have more than 5 upper soldiers. So, the maximum number of upper soldiers in a battalion is 5, and the lower is 10. Therefore, the number of battalions would be the number of ways to choose 4 upper and 8 lower soldiers. But since the upper soldiers are limited, maybe the answer is 5 choose 4 multiplied by 10 choose 8. Let me calculate that.

5 choose 4 is 5, and 10 choose 8 is 45. So 5 * 45 = 225. Therefore, the answer would be 225 different battalions. But wait, is there another way to interpret this? Maybe the problem is asking how many different battalions can be formed, considering that each battalion must have both types. But since the battle requires 4 upper and 8 lower, the number of battalions would be the number of ways to choose 4 upper and 8 lower soldiers. Which again would be 5 * 45 = 225. So, I think that's the answer. Therefore, the number of different battalions is 225.
</think>

The number of different battalions that can be sent is 225.

*⏱ Generated in 48.03 seconds*

🎉 **Amazing!** Such a tiny addition made such a big difference — the model is now reasoning step by step instead of just guessing.

This is the essence of Chain of Thought prompting: **words shape thinking**, even for AI systems.
